# 06 Compare models

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any

import polars as pl

from ixbrl_ai.display import display_wide, heading
from ixbrl_ai.test import load_population_test_results_from_mlflow


/Users/unparagoned/Code/AI_L7/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Load metrics

In [2]:
eval_data_names = {
    "LinearSVC": {"run_name":'final_model_v13_sample_10_pct_sqrt_weight',
                  "experiment_name":'model-compare'},
    "CNN": {"run_name":'candidate_v3.sample_10_pct_sqrt_weight',
            "experiment_name":'ixbrl-nn'},
    "SEC-BERT": {"run_name":'Final_model_15_epochs_sample_10_pct_sqrt_weight',
                 "experiment_name":'sentence-transformers-compare'}
}

model_results = {
    model_name: load_population_test_results_from_mlflow(experiment_name=names["experiment_name"], source_run_name=names["run_name"])[1]
    for model_name, names in eval_data_names.items()
}



2026/06/01 06:28:00 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/06/01 06:28:00 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/06/01 06:28:00 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/06/01 06:28:00 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/06/01 06:28:00 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/06/01 06:28:00 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/06/01 06:28:00 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/06/01 06:28:00 INFO alembic.runtime.migration: Will assume non-transactional DDL.


In [ ]:
subjective_inputs = {
    "LinearSVC": {
        "interpretability_explainability_notes": "High interpretability and explainability; model coefficients directly indicate how features contribute to predictions.",
        "interpretability_explainability": 5,
        "deployment_simplicity_notes": "Simple to deploy; standard libraries and low resource requirements.",
        "deployment_simplicity": 5,
        "maintenance_burden_notes": "Limited maintenance; requires monitoring, and is easy to retrain and update.",
        "maintenance_burden": 4,
        "domain_fit_notes": "Good fit for text based classification. TFIDF captures the domain specific terminology well, but may miss semantic nuances.",
        "domain_fit": 3,
        "model_lifecycle_notes": "Low risk; Mature packages with long-term support; low risk of obsolescence.",
        "model_lifecycle": 5,
        "dependency_risk_notes": "Minimal dependency risk; model can be trained using well established opensource packages.",
        "dependency_risk": 5,
    },
    "CNN": {
        "interpretability_explainability_notes": "Low interpretability; explanations require post-hoc methods such as LIME or SHAP.",
        "interpretability_explainability": 2,
        "deployment_simplicity_notes": "Moderate deployment complexity; requires GPU for optimal performance.",
        "deployment_simplicity": 3,
        "maintenance_burden_notes": "Moderate maintenance; requires monitoring and retraining.",
        "maintenance_burden": 3,
        "domain_fit_notes": "Good fit for text based classification. CNNs can capture local patterns and but can stuggle with deeper semantic relationships",
        "domain_fit": 3,
        "model_lifecycle_notes": "Low risk; Mature packages with long-term support; low risk of obsolescence.",
        "model_lifecycle": 5,
        "dependency_risk_notes": "Low dependency risk; model can be trained using well established opensource packages but the software stack is more complicated than traditional approaches.",
        "dependency_risk": 4,
    },
    "SEC-BERT": {
        "interpretability_explainability_notes": "Low interpretability; explanations require post-hoc methods such as LIME or SHAP.",
        "interpretability_explainability": 2,
        "deployment_simplicity_notes": "Complex deployment; requires significant resources, expertise and requires GPU for optimal performance.",
        "deployment_simplicity": 2,
        "maintenance_burden_notes": "Moderate maintenance; requires monitoring but reatraining can be resource intensive that can't be done on existing architecture.",
        "maintenance_burden": 2,
        "domain_fit_notes": "Theoretically a stronger fit for accountancy text data and semantic understanding, but didn't pan out in robustness testing. But was trained on US SEC data not UK company accounts data.",
        "domain_fit": 3,
        "model_lifecycle_notes": "High risk; the publically available SEC-BERT has received limited maintenance since release creating unvertainty around future support and updates",
        "model_lifecycle": 1,
        "dependency_risk_notes": "High dependency risk; reliance on unverified external third-party pre-trained models.",
        "dependency_risk": 2,
    },
}


metric_config = {
    "accuracy": {
        "weight": 0.20,
        "direction": "higher",
        "source": "objective",
        "description": "Overall accuracy; intuitive but can be misleading with imbalance.",
    },
    "f1_macro": {
        "weight": 0.25,
        "direction": "higher",
        "source": "objective",
        "description": "Macro F1; important for class imbalance.",
    },
    "recall_macro": {
        "weight": 0.05,
        "direction": "higher",
        "source": "objective",
        "description": "Macro recall; penalises false negatives across classes.",
    },
    "precision_macro": {
        "weight": 0.05,
        "direction": "higher",
        "source": "objective",
        "description": "Macro precision; penalises false positives across classes.",
    },
    "f1_weighted": {
        "weight": 0.10,
        "direction": "higher",
        "source": "objective",
        "description": "Weighted F1; reflects population-level performance.",
    },
    "train_time": {
        "weight": 0.10,
        "direction": "lower",
        "source": "objective",
        "description": "Training time in seconds.",
    },
    "inference_time": {
        "weight": 0.10,
        "direction": "lower",
        "source": "objective",
        "description": "Inference time per sample in seconds.",
    },
    "model_size": {
        "weight": 0.10,
        "direction": "lower",
        "source": "objective",
        "description": "Model size in bytes.",
    },
    "interpretability": {
        "weight": 0.10,
        "direction": "higher",
        "source": "subjective",
        "description": "Ease of explaining model behaviour.",
    },
    "deployment_simplicity": {
        "weight": 0.10,
        "direction": "higher",
        "source": "subjective",
        "description": "Ease of deployment and operationalisation.",
    },
    "maintenance_burden": {
        "weight": 0.10,
        "direction": "higher",
        "source": "subjective",
        "description": "Ease of maintaining the model over time.",
    },
    "domain_fit": {
        "weight": 0.10,
        "direction": "higher",
        "source": "subjective",
        "description": "Fit for domain-specific requirements",
    },
    "model_lifecycle": {
        "weight": 0.10,
        "direction": "higher",
        "source": "subjective",
        "description": "Expected longevity and support for the model.",
    },
    "dependency_risk": {
        "weight": 0.15,
        "direction": "higher", 
        "source": "subjective",
        "description": "Risk associated with external dependencies.",
    },     
}


def confidence_adjustment_expr(metric_name: str) -> pl.Expr:
    mean = pl.col(metric_name)
    low = pl.col(f"{metric_name}_ci_low")
    high = pl.col(f"{metric_name}_ci_high")

    best_mean = mean.max()
    best_low = low.filter(mean == best_mean).first()

    # If a model's CI overlaps the best model's CI, reduce confidence.
    # 1.0 = clearly separated from weaker models
    # near 0.0 = not meaningfully distinguishable
    return pl.when(high >= best_low).then(0.35).otherwise(1.0).alias(f"{metric_name}_confidence_factor")


def extract_holdout_metrics(
    model_name: str,
    result_object: dict[str, Any],
    dataset_key: str = "holdout_10k",
) -> dict[str, Any]:
    """Extracts the mean and confidence interval for each metric from the holdout results of a model.

    Args:
        model_name: The name of the model.
        result_object: The dictionary containing the results for the model.
        dataset_key: The key in the result_object that corresponds to the holdout dataset results.

    Returns:
        A dictionary containing the model name, mean, confidence interval low, confidence interval high, and confidence interval width for each metric.
    """

    holdout = result_object[dataset_key]

    row = {"model": model_name}

    for metric_name, metric_payload in holdout.items():
        row[metric_name] = metric_payload["mean"]
        row[f"{metric_name}_ci_low"] = metric_payload["confidence_interval"]["low"]
        row[f"{metric_name}_ci_high"] = metric_payload["confidence_interval"]["high"]
        row[f"{metric_name}_ci_width"] = (
            metric_payload["confidence_interval"]["high"] - metric_payload["confidence_interval"]["low"]
        )

    if "train_time" in result_object:
        row["train_time"] = result_object["train_time"]
    
    if "model_size" in result_object:
        row["model_size"] = result_object["model_size"]

    if "inference_time" in result_object:
        row["inference_time"] = result_object["inference_time"]

    return row


def min_max_score_expr(metric_name: str, direction: str) -> pl.Expr:
    """Creates a Polars expression to calculate a min-max normalised score for a given metric.

    Args:
        metric_name: The name of the metric to score.
        direction: The direction of the metric, either "higher" or "lower".

    Returns:
        A Polars expression that calculates the min-max normalised score for the metric.
    """
    value = pl.col(metric_name)
    minimum = value.min()
    maximum = value.max()
    denominator = maximum - minimum

    if direction == "higher":
        score = (value - minimum) / denominator
    elif direction == "lower":
        score = (maximum - value) / denominator
    else:
        raise ValueError(f"Unsupported direction: {direction}")

    return pl.when(denominator == 0).then(1.0).otherwise(score).alias(f"{metric_name}_score")


def build_decision_matrix(
    model_results: dict[str, dict[str, Any]],
    subjective_inputs: dict[str, dict[str, Any]],
    metric_config: dict[str, dict[str, Any]],
    dataset_key: str = "holdout_10k",
    overlap_penalty: float = 0.35,
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Builds a decision matrix comparing multiple models across objective and subjective metrics.
    Args:
        model_results: A dictionary mapping model names to their result objects containing metric values and confidence intervals.
        subjective_inputs: A dictionary mapping model names to their subjective metric ratings and notes.
        metric_config: A dictionary defining the configuration for each metric, including weight, direction, source, and description.
        dataset_key: The key in the result objects that corresponds to the holdout dataset results.
        overlap_penalty: The penalty factor to apply to scores of models whose confidence intervals overlap with the best model.
    Returns:
        A tuple containing:
            - A Polars DataFrame representing the decision matrix with all metrics, scores, confidence factors, and the final decision score for each model.
            - A long-format Polars DataFrame suitable for visualization, with columns for model, metric, and value.
    """


    objective_rows = [
        extract_holdout_metrics(model_name, result_object, dataset_key)
        for model_name, result_object in model_results.items()
    ]

    objective_df = pl.DataFrame(objective_rows)

    subjective_df = pl.DataFrame([{"model": model_name, **values} for model_name, values in subjective_inputs.items()])

    matrix = objective_df.join(subjective_df, on="model", how="left")

    numeric_metrics = [
        metric_name
        for metric_name in metric_config
        if metric_name in matrix.columns and matrix[metric_name].dtype.is_numeric()
    ]

    metrics_with_ci = [
        metric_name
        for metric_name in numeric_metrics
        if (f"{metric_name}_ci_low" in matrix.columns and f"{metric_name}_ci_high" in matrix.columns)
    ]

    score_exprs = [
        min_max_score_expr(
            metric_name=metric_name,
            direction=metric_config[metric_name]["direction"],
        )
        for metric_name in numeric_metrics
    ]

    matrix = matrix.with_columns(score_exprs)

    confidence_factor_exprs = []

    for metric_name in metrics_with_ci:
        direction = metric_config[metric_name]["direction"]

        if direction == "higher":
            best_value = pl.col(metric_name).max()
            best_ci_low = pl.col(f"{metric_name}_ci_low").filter(pl.col(metric_name) == best_value).first()

            confidence_factor = (
                pl.when(pl.col(f"{metric_name}_ci_high") >= best_ci_low).then(overlap_penalty).otherwise(1.0)
            )

        elif direction == "lower":
            best_value = pl.col(metric_name).min()
            best_ci_high = pl.col(f"{metric_name}_ci_high").filter(pl.col(metric_name) == best_value).first()

            confidence_factor = (
                pl.when(pl.col(f"{metric_name}_ci_low") <= best_ci_high).then(overlap_penalty).otherwise(1.0)
            )

        else:
            raise ValueError(f"Unsupported direction: {direction}")

        confidence_factor_exprs.append(confidence_factor.alias(f"{metric_name}_confidence_factor"))

    if confidence_factor_exprs:
        matrix = matrix.with_columns(confidence_factor_exprs)

    weighted_score_exprs = []

    for metric_name in numeric_metrics:
        confidence_factor = (
            pl.col(f"{metric_name}_confidence_factor") if metric_name in metrics_with_ci else pl.lit(1.0)
        )

        weighted_score_exprs.append(
            (pl.col(f"{metric_name}_score") * confidence_factor * metric_config[metric_name]["weight"]).alias(
                f"{metric_name}_weighted_score"
            )
        )

    matrix = matrix.with_columns(weighted_score_exprs)

    total_weight = sum(metric_config[metric_name]["weight"] for metric_name in numeric_metrics)

    matrix = matrix.with_columns(
        (sum(pl.col(f"{metric_name}_weighted_score") for metric_name in numeric_metrics) / total_weight).alias(
            "decision_score"
        )
    ).sort("decision_score", descending=True)

    long_matrix = matrix.unpivot(
        index="model",
        variable_name="metric",
        value_name="value",
    )

    return matrix, long_matrix



decision_matrix, decision_matrix_long = build_decision_matrix(
    model_results=model_results,
    subjective_inputs=subjective_inputs,
    metric_config=metric_config,
    dataset_key="holdout_10k",
)


def build_subjective_notes_table(
    subjective_inputs: dict[str, dict[str, Any]],
) -> pl.DataFrame:
    """Builds a Polars DataFrame containing the subjective notes for each model and metric.
    Args:
        subjective_inputs: A dictionary mapping model names to their subjective metric ratings and notes.
    Returns:
        A Polars DataFrame where each row corresponds to a model and columns contain the subjective notes for each metric.
    """
    note_rows = []

    for model_name, values in subjective_inputs.items():
        row = {"model": model_name}

        for key, value in values.items():
            if key.endswith("_notes"):
                clean_key = key.removesuffix("_notes")
                row[clean_key] = value

        note_rows.append(row)

    return pl.DataFrame(note_rows)


subjective_notes_table = build_subjective_notes_table(subjective_inputs)


def build_raw_values_table(
    decision_matrix: pl.DataFrame,
    metric_config: dict[str, dict[str, Any]],
) -> pl.DataFrame:
    """Builds a Polars DataFrame containing the raw values for each model and metric.

    Args:
        decision_matrix: A Polars DataFrame containing the decision matrix with all metrics, scores, and confidence factors.
        metric_config: A dictionary defining the configuration for each metric, including weight, direction, source, and description.

    Returns:
        A Polars DataFrame where each row corresponds to a model and columns contain the raw values for each metric.
    """
    expressions = [pl.col("model")]

    for metric_name, config in metric_config.items():

        if metric_name not in decision_matrix.columns:
            continue

        has_ci = (
            f"{metric_name}_ci_low" in decision_matrix.columns
            and f"{metric_name}_ci_high" in decision_matrix.columns
        )

        if has_ci:
            expressions.append(
                pl.concat_str(
                    [
                        pl.col(metric_name).round(3).cast(pl.String),
                        pl.lit(" (CI "),
                        pl.col(f"{metric_name}_ci_low").round(3).cast(pl.String),
                        pl.lit("–"),
                        pl.col(f"{metric_name}_ci_high").round(3).cast(pl.String),
                        pl.lit(")"),
                    ]
                ).alias(metric_name)
            )
        else:
            expressions.append(pl.col(metric_name))

    return decision_matrix.select(expressions)


raw_values_table = build_raw_values_table(
    decision_matrix=decision_matrix,
    metric_config=metric_config,
)


def build_weighting_table(
    metric_config: dict[str, dict[str, Any]],
) -> pl.DataFrame:
    """Builds a Polars DataFrame containing the weighting information for each metric.

    Args:
        metric_config: A dictionary defining the configuration for each metric, including weight, direction, source, and description.

    Returns:
        A Polars DataFrame where each row corresponds to a metric and columns contain the weighting information.
    """
    return (
        pl.DataFrame(
            [
                {
                    "metric": metric_name,
                    "weight": config["weight"],
                    "direction": config["direction"],
                    "source": config["source"],
                    "description": config["description"],
                }
                for metric_name, config in metric_config.items()
            ]
        )
        .with_columns(
            (pl.col("weight") / pl.col("weight").sum()).alias("normalised_weight")
        )
        .sort("normalised_weight", descending=True)
    )


weighting_table = build_weighting_table(metric_config)


def build_final_scores_table(
    decision_matrix: pl.DataFrame,
    metric_config: dict[str, dict[str, Any]],
) -> pl.DataFrame:
    """Builds a Polars DataFrame containing the final decision scores for each model.
    Args:
        decision_matrix: A Polars DataFrame containing the decision matrix with all metrics, scores, and confidence factors.
        metric_config: A dictionary defining the configuration for each metric, including weight, direction, source, and description.
    Returns:
        A Polars DataFrame where each row corresponds to a model and columns contain the final decision score and weighted scores for each metric.
    """

    weighted_score_columns = [
        f"{metric_name}_weighted_score"
        for metric_name in metric_config
        if f"{metric_name}_weighted_score" in decision_matrix.columns
    ]

    return decision_matrix.select(
        [
            "model",
            "decision_score",
            *weighted_score_columns,
        ]
    ).sort("decision_score", descending=True)


final_scores_table = build_final_scores_table(
    decision_matrix=decision_matrix,
    metric_config=metric_config,
)

heading("2. Model Comparison Decision Matrix")
heading("2.1 Subjective Notes", level=2)
display_wide(subjective_notes_table)
heading("2.2 Raw Values", level=2)
display_wide(raw_values_table)
heading("2.3 Weighting", level=2)
display_wide(weighting_table)
heading("2.4 Final Scores", level=2)
display_wide(final_scores_table)

## 2. Model Comparison Decision Matrix

## 2.1 Subjective Notes

model,interpretability_explainability,deployment_simplicity,maintenance_burden,domain_fit,model_lifecycle,dependency_risk
str,str,str,str,str,str,str
"""LinearSVC""","""High interpretability and explainability; model coefficients directly indicate how features contribute to predictions.""","""Simple to deploy; standard libraries and low resource requirements.""","""Limited maintenance; requires monitoring, and is easy to retrain and update.""","""Good fit for text based classification. TFIDF captures the domain specific terminology well, but may miss semantic nuances.""","""Low risk; Mature packages with long-term support; low risk of obsolescence.""","""Minimal dependency risk; model can be trained using well established opensource packages."""
"""CNN""","""Low interpretability; explanations require post-hoc methods such as LIME or SHAP.""","""Moderate deployment complexity; requires GPU for optimal performance.""","""Moderate maintenance; requires monitoring and retraining.""","""Good fit for text based classification. CNNs can capture local patterns and but can stuggle with deeper semantic relationships""","""Low risk; Mature packages with long-term support; low risk of obsolescence.""","""Low dependency risk; model can be trained using well established opensource packages but the software stack is more complicated than traditional approaches."""
"""SEC-BERT""","""Low interpretability; explanations require post-hoc methods such as LIME or SHAP.""","""Complex deployment; requires significant resources, expertise and requires GPU for optimal performance.""","""Moderate maintenance; requires monitoring but reatraining can be resource intensive that can't be done on existing architecture.""","""Theoretically a stronger fit for accountancy text data and semantic understanding, but didn't pan out in robustness testing. But was trained on US SEC data not UK company accounts data.""","""High risk; the publically available SEC-BERT has received limited maintenance since release creating unvertainty around future support and updates""","""High dependency risk; reliance on unverified external third-party pre-trained models."""


## 2.2 Raw Values

model,accuracy,f1_macro,recall_macro,precision_macro,f1_weighted,train_time,inference_time,model_size,deployment_simplicity,maintenance_burden,domain_fit,model_lifecycle,dependency_risk
str,str,str,str,str,str,i64,f64,i64,i64,i64,i64,i64,i64
"""LinearSVC""","""0.975 (CI 0.972–0.978)""","""0.8 (CI 0.781–0.829)""","""0.821 (CI 0.806–0.856)""","""0.823 (CI 0.792–0.847)""","""0.971 (CI 0.967–0.974)""",144,0.640634,8126919,5,4,3,5,5
"""CNN""","""0.977 (CI 0.974–0.98)""","""0.808 (CI 0.788–0.84)""","""0.825 (CI 0.81–0.863)""","""0.829 (CI 0.799–0.856)""","""0.972 (CI 0.969–0.976)""",2640,23.86633,29723312,3,3,3,5,4
"""SEC-BERT""","""0.977 (CI 0.974–0.98)""","""0.823 (CI 0.798–0.85)""","""0.834 (CI 0.817–0.867)""","""0.855 (CI 0.814–0.873)""","""0.973 (CI 0.969–0.976)""",4023,143.107253,1756868591,2,2,3,1,2


## 2.3 Weighting

metric,weight,direction,source,description,normalised_weight
str,f64,str,str,str,f64
"""f1_macro""",0.25,"""higher""","""objective""","""Macro F1; important for class imbalance.""",0.15625
"""accuracy""",0.2,"""higher""","""objective""","""Overall accuracy; intuitive but can be misleading with imbalance.""",0.125
"""dependency_risk""",0.15,"""higher""","""subjective""","""Risk associated with external dependencies.""",0.09375
"""f1_weighted""",0.1,"""higher""","""objective""","""Weighted F1; reflects population-level performance.""",0.0625
"""train_time""",0.1,"""lower""","""objective""","""Training time in seconds.""",0.0625
"""inference_time""",0.1,"""lower""","""objective""","""Inference time per sample in seconds.""",0.0625
"""model_size""",0.1,"""lower""","""objective""","""Model size in bytes.""",0.0625
"""interpretability""",0.1,"""higher""","""subjective""","""Ease of explaining model behaviour.""",0.0625
"""deployment_simplicity""",0.1,"""higher""","""subjective""","""Ease of deployment and operationalisation.""",0.0625


## 2.4 Final Scores

model,decision_score,accuracy_weighted_score,f1_macro_weighted_score,recall_macro_weighted_score,precision_macro_weighted_score,f1_weighted_weighted_score,train_time_weighted_score,inference_time_weighted_score,model_size_weighted_score,deployment_simplicity_weighted_score,maintenance_burden_weighted_score,domain_fit_weighted_score,model_lifecycle_weighted_score,dependency_risk_weighted_score
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""LinearSVC""",0.566667,0.0,0.0,0.0,0.0,0.0,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.15
"""CNN""",0.475507,0.048696,0.028617,0.005885,0.003167,0.025448,0.035654,0.083697,0.098765,0.033333,0.05,0.1,0.1,0.1
"""SEC-BERT""",0.218333,0.07,0.0875,0.0175,0.0175,0.035,0.0,0.0,0.0,0.0,0.0,0.1,0.0,0.0


## 2.5 Rubric
### Interpretability & Explainability

| Score | Definition |
|---------|---------|
| 1 | Model behaviour is largely opaque and explanations are difficult to generate or validate. |
| 2 | Model behaviour is not directly interpretable and requires post-hoc explanation techniques (e.g., SHAP, LIME) to provide insight into predictions. |
| 3 | Some internal model components can be inspected, providing limited understanding of prediction behaviour. |
| 4 | Important decision factors are generally understandable and can be communicated with moderate effort. |
| 5 | Model behaviour is intrinsically interpretable, with predictions directly traceable to understandable features or rules. |

### Deployment Simplicity

| Score | Definition |
|---------|---------|
| 1 | Requires specialised infrastructure, significant engineering effort, and extensive deployment expertise. |
| 2 | Requires substantial computational resources and complex deployment procedures. |
| 3 | Moderate deployment complexity with some specialised requirements. |
| 4 | Straightforward deployment using commonly available tooling and infrastructure. |
| 5 | Minimal infrastructure requirements and simple deployment using standard software components. |

### Maintenance Burden

| Score | Definition |
|---------|---------|
| 1 | High maintenance burden requiring frequent updates, specialist knowledge, and ongoing external dependencies. |
| 2 | Significant monitoring and maintenance effort required, including model lifecycle management. |
| 3 | Moderate maintenance effort with periodic monitoring and updates. |
| 4 | Limited maintenance requirements with well-understood operational procedures. |
| 5 | Minimal maintenance burden due to model simplicity, stability, and mature tooling. |

### Domain Fit

| Score | Definition |
|---------|---------|
| 1 | Poor alignment with domain requirements and expected data characteristics. |
| 2 | Limited suitability for the target domain with significant known weaknesses. |
| 3 | Adequate suitability for the target domain and capable of addressing core requirements. |
| 4 | Strong alignment with domain requirements and data characteristics. |
| 5 | Excellent alignment with domain-specific language, concepts, and classification objectives. |

### Model Lifecycle & Sustainability

| Score | Definition |
|---------|---------|
| 1 | Model is effectively abandoned or dependent on unsupported external resources. |
| 2 | Limited evidence of ongoing maintenance or future development. |
| 3 | Some evidence of continued support but long-term sustainability uncertain. |
| 4 | Actively maintained with a clear update path. |
| 5 | Mature technology with strong ecosystem support and multiple replacement options. |

### Dependency Risk

| Score | Definition |
|---------|---------|
| 1 | Significant dependence on poorly documented or externally controlled components with limited transparency or support. |
| 2 | High reliance on external models, datasets, or providers, creating notable governance and maintenance risks. |
| 3 | Moderate external dependencies with manageable governance concerns. |
| 4 | Limited dependency risk with mature and well-supported components. |
| 5 | Minimal governance risk due to transparent, internally controlled, and well-established technologies. |